# Databricks/PySpark dla Data Analyst/Engineer — Moduł 4: Joiny w Sparku

W Module 3 łączyliśmy tabele "na skróty", zanim formalnie omówiliśmy `join`. Ten moduł
nadrabia to porządnie: typy joinów, semi/anti joiny, broadcast join i pierwsze spojrzenie
na plan wykonania (`explain()`) w kontekście łączenia tabel.

## Spis treści
1. [Typy joinów: inner/left/right/outer](#sec1)
2. [Semi i anti joiny](#sec2)
3. [Broadcast join i koszt shuffle](#sec3)
4. [Czytanie planu wykonania joinów](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("kurs_spark").getOrCreate()
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Typy joinów: inner/left/right/outer

Składnia `df1.join(df2, warunek, typ)` — dokładnie te same typy joinów co w SQL-u
(Moduł 5 kursu SQL), tylko teraz jako metoda DataFrame. Gdy kolumna łącząca nazywa się
tak samo w obu tabelach, można podać samą nazwę jako string zamiast pełnego warunku.

In [ ]:
# Domyslnie: inner join -- tylko wiersze z dopasowaniem po obu stronach
zamowienia.join(klienci, "klient_id").select(
    "zamowienie_id", "klient_id", "miasto", "segment", "status"
).show(5)


In [ ]:
# left join -- wszystkie wiersze z lewej tabeli, nawet bez dopasowania (NULL po prawej)
pracownicy.join(zamowienia, "pracownik_id", "left").select(
    "pracownik_id", "region", "zamowienie_id"
).show(5)


Gdy nazwy kolumn łączących RÓŻNIĄ się między tabelami, trzeba podać pełny warunek jako
wyrażenie logiczne (a nie string) -- wtedy obie kolumny zostają w wyniku (często trzeba
później jedną z nich usunąć przez `.drop()`).

In [ ]:
# Warunek jako wyrazenie -- przydatne, gdy nazwy kolumn sie roznia
zamowienia.join(
    klienci, zamowienia.klient_id == klienci.klient_id, "inner"
).drop(klienci.klient_id).select("zamowienie_id", "klient_id", "miasto").show(5)


> ⚠️ **right i full outer -- rzadziej używane, ale warto znać**
>
> `"right"` to left join z odwróconymi tabelami (rzadko piszemy go bezpośrednio -- łatwiej zamienić kolejność `df1`/`df2` i użyć `"left"`). `"outer"` (albo `"full"`) zwraca wszystkie wiersze z OBU stron, z `NULL` tam, gdzie brak dopasowania po drugiej stronie -- rzadki przypadek w praktyce, ale przydatny do wykrywania rozbieżności między dwoma źródłami danych.

<a id="sec2"></a>
## 2. Semi i anti joiny

To dwa typy joinów, których NIE ma w klasycznym SQL-u pod tą nazwą (w SQL-u osiąga się je
przez `WHERE EXISTS`/`WHERE NOT EXISTS`, jak w Module 6 kursu SQL) -- w Sparku są
wbudowanymi typami `join()`. Kluczowa cecha: zwracają kolumny TYLKO z lewej tabeli,
nawet jeśli warunek dotyczy prawej.

In [ ]:
# left_semi -- klienci, ktorzy MAJA przynajmniej jedno zamowienie (tylko kolumny klienci)
klienci.join(zamowienia, "klient_id", "left_semi").show(5)


In [ ]:
# left_anti -- klienci, ktorzy NIGDY nie zlozyli zamowienia (tylko kolumny klienci)
klienci_bez_zamowien = klienci.join(zamowienia, "klient_id", "left_anti")
print(f"Klienci bez zamowien: {klienci_bez_zamowien.count()}")
klienci_bez_zamowien.show(5)


> ⚡ **Semi/anti join zamiast join + filter + drop**
>
> Zanim poznaliśmy `left_semi`/`left_anti`, to samo dałoby się osiągnąć przez zwykły `join` i odfiltrowanie wierszy z `NULL`-em (albo bez) po prawej stronie -- ale to wymagałoby dociągnięcia WSZYSTKICH kolumn prawej tabeli, tylko po to, żeby je zaraz odrzucić. Semi/anti join od razu mówi Sparkowi, że interesują nas TYLKO kolumny lewej strony -- to zarówno czytelniejszy kod, jak i zwykle szybszy plan wykonania.

<a id="sec3"></a>
## 3. Broadcast join i koszt shuffle

Domyślnie, żeby połączyć dwie duże tabele, Spark musi wykonać **shuffle** -- przetasować
dane między executorami tak, żeby wiersze z tym samym kluczem trafiły na tę samą maszynę
(**SortMergeJoin**). To kosztowna operacja sieciowa. Jeśli JEDNA z tabel jest MAŁA (mieści
się w pamięci pojedynczego executora), Spark może zamiast tego wysłać jej KOPIĘ do
KAŻDEGO executora (**BroadcastHashJoin**) -- unikając shuffle'a całkowicie.

In [ ]:
from pyspark.sql import functions as F

# produkty (40 wierszy) to idealny kandydat na broadcast -- laczymy z duzo wieksza
# tabela pozycje_zamowien
polaczone = pozycje_zamowien.join(F.broadcast(produkty), "produkt_id")
polaczone.select("pozycja_id", "produkt_id", "kategoria", "cena").show(5)


> ⚡ **Spark zwykle robi to automatycznie**
>
> Spark ma wbudowany próg (`spark.sql.autoBroadcastJoinThreshold`, domyślnie 10 MB) -- jeśli jedna z tabel jest mniejsza, Catalyst SAM zamieni join na broadcast, bez żadnej podpowiedzi z naszej strony. `F.broadcast()` przydaje się głównie wtedy, gdy Spark błędnie oszacuje rozmiar tabeli (np. po serii transformacji) i sam by tego nie wybrał -- wtedy podpowiadamy mu to jawnie.

> ⚠️ **Broadcast dużej tabeli to pomyłka, nie optymalizacja**
>
> `F.broadcast()` na tabeli, która NIE mieści się wygodnie w pamięci pojedynczego executora, może doprowadzić do błędu braku pamięci (`OutOfMemoryError`) albo poważnego spowolnienia -- ta podpowiedź ma sens tylko dla naprawdę MAŁYCH tabel (słowniki, tabele wymiarów typu `produkty`/`pracownicy`), nigdy dla głównych tabel faktów typu `zamowienia`/`pozycje_zamowien`.

<a id="sec4"></a>
## 4. Czytanie planu wykonania joinów

`.explain()` (wprowadzone w Module 1) pokazuje, JAKI typ joina Spark faktycznie wybrał --
warto to sprawdzić, gdy wydajność joina jest podejrzana.

In [ ]:
polaczone.explain()


W wypisanym planie szukaj słów `BroadcastHashJoin` (szybko, bez shuffle'a) albo
`SortMergeJoin` (wolniej, wymaga shuffle'a obu stron). Dokładne odczytywanie pełnych
planów wykonania (fizyczny plan, statystyki, etapy) wracamy szerzej w Module 9
(optymalizacja) -- na razie wystarczy umieć rozpoznać te dwie nazwy.

> 🧱 **Spark UI w Databricks**
>
> W prawdziwym środowisku Databricks, oprócz `.explain()` w notatniku, masz dostęp do graficznego **Spark UI** -- zakładki 'SQL / DataFrame' pokazującej te same plany wizualnie, z czasami wykonania każdego etapu. Lokalnie w tym kursie korzystamy wyłącznie z tekstowego `.explain()`.

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- typy joinów: `inner`, `left`, `right`, `outer` -- składnia `df1.join(df2, warunek, typ)`,
- `left_semi` i `left_anti` -- filtrowanie po istnieniu dopasowania, bez dociągania kolumn drugiej strony,
- broadcast join (`F.broadcast()`) jako sposób na uniknięcie kosztownego shuffle'a dla małych tabel,
- odczytywanie z `.explain()`, czy Spark wybrał `BroadcastHashJoin` czy `SortMergeJoin`.

### 📝 Ćwiczenie 1: Zamówienia z danymi pracownika

Połącz `zamowienia` z `pracownicy` (inner join po `pracownik_id`) i wyświetl `zamowienie_id`, `region`, `dzial`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zamowienia.join(pracownicy, "pracownik_id").select(
    "zamowienie_id", "region", "dzial"
).show(5)
```

</details>

### 📝 Ćwiczenie 2: Produkty, które nigdy nie zostały zamówione

Użyj `left_anti` joina między `produkty` a `pozycje_zamowien`, żeby znaleźć produkty, które nigdy nie pojawiły się w żadnym zamówieniu.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
niezamawiane = produkty.join(pozycje_zamowien, "produkt_id", "left_anti")
print(f"Produkty bez zamowien: {niezamawiane.count()}")
niezamawiane.show()
```

</details>

### 📝 Ćwiczenie 3: Broadcast join z pracownikami

Połącz `zamowienia` z `F.broadcast(pracownicy)` (pracownicy to mała tabela) i sprawdź w `.explain()`, czy Spark użył `BroadcastHashJoin`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql import functions as F

wynik = zamowienia.join(F.broadcast(pracownicy), "pracownik_id")
wynik.explain()
```

</details>

### 📝 Ćwiczenie 4: Klienci Premium bez zamówień Dostarczonych

Znajdź klientów z segmentu `'Premium'`, którzy NIE MAJĄ żadnego zamówienia ze statusem `'Dostarczone'` (podpowiedź: najpierw odfiltruj `zamowienia` do statusu `'Dostarczone'`, potem zrób `left_anti` z `klienci`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dostarczone = zamowienia.filter(zamowienia.status == "Dostarczone")
premium = klienci.filter(klienci.segment == "Premium")
premium.join(dostarczone, "klient_id", "left_anti").show()
```

</details>

> 🔥 **Wyzwanie: pełny raport z trzema joinami i broadcastem**
>
> Połącz `zamowienia`, `klienci` i `pozycje_zamowien` (z `F.broadcast(produkty)` na koniec dla kategorii), oblicz `wartosc_pozycji` (jak w Module 2/3), zagreguj sumę wartości per `(segment, kategoria)`, i wyświetl `.explain()` całego łańcucha przed wywołaniem `.show()`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql import functions as F

polaczone = (
    zamowienia.join(klienci, "klient_id")
    .join(pozycje_zamowien, "zamowienie_id")
    .join(F.broadcast(produkty), "produkt_id")
    .withColumn("wartosc_pozycji", F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat")))
)

wynik = polaczone.groupBy("segment", "kategoria").agg(
    F.round(F.sum("wartosc_pozycji"), 2).alias("suma_wartosci")
).orderBy(F.col("suma_wartosci").desc())

wynik.explain()
wynik.show(20)
```

Zauwaz w planie, ze tylko join z produkty (broadcast) unika shuffle'a -- polaczenia z klienci i pozycje_zamowien to duze tabele po obu stronach, wiec Spark i tak wykona tam SortMergeJoin.

</details>

## Co dalej?

W **Module 5** poznamy Spark SQL -- jak zarejestrować DataFrame jako tymczasowy widok
(`createOrReplaceTempView`) i pisać dokładnie takie same zapytania SQL jak w kursie SQL,
tylko teraz wewnątrz notatnika Spark.